# Notebook 06: Comparação Final de Modelos

Este notebook junta os principais agentes do projeto num torneio final e usa a versão modular atual do `AlphaZero` na comparação.


## Objetivo

O objetivo deste notebook é:

- carregar as melhores versões disponíveis dos agentes;
- correr um torneio `round-robin`;
- calcular métricas agregadas e `Elo`;
- produzir uma leitura comparativa final;
- apoiar diretamente a discussão dos resultados no relatório.

## Passo 1: Preparação do ambiente

Importamos os agentes e utilitários necessários para o torneio comparativo final do projeto. O `AlphaZero` carregado aqui já usa a nova arquitetura residual com `n_filters` e `n_res_blocks`.


In [ ]:
from __future__ import annotations

import json
import statistics
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config.yaml").exists():
    raise FileNotFoundError("Could not find project root containing config.yaml")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUTS = ROOT / "outputs"

from connect4_rl.config import load_config
CONFIG = load_config(ROOT / "config.yaml")

NOTEBOOK_DEVICE = CONFIG.resolve_device()
if NOTEBOOK_DEVICE == "cuda":
    torch.set_default_device(NOTEBOOK_DEVICE)
print(
    {
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_count": torch.cuda.device_count(),
        "device": NOTEBOOK_DEVICE,
        "cuda_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
)
OUTPUTS

from connect4_rl.agents.baselines import HeuristicAgent, RandomAgent
from connect4_rl.agents.planning import MCTSAgent
from connect4_rl.envs.connect_four import apply_action, initial_state, is_terminal, legal_actions, render_ascii
from connect4_rl.experiments import build_agent_factory_from_run, build_reference_factory, compute_elo_ratings, find_best_run, round_robin, round_robin_detailed


## Passo 2: Configuração da comparação final

Nesta célula definimos o comportamento do torneio comparativo.

- `run_comparison_tournament`: se estiver a `True`, corre o torneio final.
- `games_per_pair`: número de jogos por par de agentes.
- `mcts_simulations`: orçamento de procura do baseline `MCTS`.
- os checkpoints dos agentes aprendidos são escolhidos a partir das métricas guardadas em `outputs/`.


In [ ]:
run_comparison_tournament = False

seed = CONFIG.notebook_settings.seed
games_per_pair = CONFIG.notebook_settings.model_comparison_games_per_pair
mcts_simulations = CONFIG.notebook_settings.model_comparison_mcts_simulations
include_heuristic_reference = True

validation_games_against_random = CONFIG.notebook_settings.model_comparison_validation_games
validation_threshold_against_random = CONFIG.notebook_settings.model_comparison_validation_threshold

dqn_run = find_best_run(OUTPUTS, "dqn")
ppo_run = find_best_run(OUTPUTS, "ppo")
alphazero_run = find_best_run(OUTPUTS, "alphazero")
{
    "dqn": dqn_run.metrics_path.name if dqn_run else None,
    "ppo": ppo_run.metrics_path.name if ppo_run else None,
    "alphazero": alphazero_run.metrics_path.name if alphazero_run else None,
}


## Passo 3: Construção dos agentes do torneio

Aqui montamos o conjunto de agentes que entram no comparativo final, incluindo `MCTS`, `DQN`, `PPO` e `AlphaZero` quando os seus checkpoints existem.

No caso do `AlphaZero`, os hiperparâmetros estruturais são lidos do `config` guardado junto do treino, para garantir compatibilidade com o checkpoint.


In [ ]:
comparison_factories = {
    "mcts": build_reference_factory("mcts", seed=seed, mcts_simulations=mcts_simulations),
}

if include_heuristic_reference:
    comparison_factories["heuristic"] = build_reference_factory("heuristic", seed=seed)

if dqn_run is not None:
    comparison_factories["dqn"] = build_agent_factory_from_run(dqn_run, root=ROOT, device=NOTEBOOK_DEVICE)

if ppo_run is not None:
    comparison_factories["ppo"] = build_agent_factory_from_run(ppo_run, root=ROOT, device=NOTEBOOK_DEVICE)

if alphazero_run is not None:
    comparison_factories["alphazero"] = build_agent_factory_from_run(alphazero_run, root=ROOT, device=NOTEBOOK_DEVICE)

list(comparison_factories.keys())


## Passo 4: Execução do torneio round-robin

Se a opção correspondente estiver ativa, esta célula corre o torneio `round-robin` entre os agentes carregados.

In [ ]:
# Resultados vazios até o torneio ser efetivamente executado.
comparison_results = {}
comparison_matches = []

if run_comparison_tournament:
    comparison_results, comparison_matches = round_robin_detailed(comparison_factories, games_per_pair=games_per_pair)
    comparison_results
else:
    print("Set run_comparison_tournament = True to run the final comparison.")


## Passo 5: Análise dos resultados do torneio

Nesta célula transformamos os resultados do torneio em gráficos e produzimos um ranking simples dos agentes.

In [ ]:
if comparison_results:
    labels = list(comparison_results.keys())
    win_rates = [float(comparison_results[name].get("win_rate", 0.0)) for name in labels]
    draw_rates = [float(comparison_results[name].get("draw_rate", 0.0)) for name in labels]
    wins = [int(comparison_results[name].get("wins", 0)) for name in labels]
    losses = [int(comparison_results[name].get("losses", 0)) for name in labels]

    fig, axes = plt.subplots(1, 3, figsize=(17, 4))

    axes[0].bar(labels, win_rates, color="#2a9d8f")
    axes[0].set_ylim(0.0, 1.0)
    axes[0].set_title("Taxa de vitória no torneio final")
    axes[0].set_ylabel("Taxa de vitória")
    axes[0].tick_params(axis="x", rotation=20)
    for idx, value in enumerate(win_rates):
        axes[0].text(idx, value + 0.02, f"{value:.2f}", ha="center")

    axes[1].bar(labels, draw_rates, color="#e9c46a")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_title("Taxa de empates no torneio final")
    axes[1].set_ylabel("Taxa de empates")
    axes[1].tick_params(axis="x", rotation=20)
    for idx, value in enumerate(draw_rates):
        axes[1].text(idx, value + 0.02, f"{value:.2f}", ha="center")

    x = list(range(len(labels)))
    width = 0.35
    axes[2].bar([value - width / 2 for value in x], wins, width=width, label="Vitórias", color="#457b9d")
    axes[2].bar([value + width / 2 for value in x], losses, width=width, label="Derrotas", color="#e76f51")
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(labels, rotation=20)
    axes[2].set_title("Vitórias e derrotas por modelo")
    axes[2].set_ylabel("Número de jogos")
    axes[2].legend()

    fig.tight_layout()
    plt.show()

ranking = sorted(
    comparison_results.items(),
    key=lambda item: (item[1]["win_rate"], item[1]["draw_rate"]),
    reverse=True,
)
ranking


## Passo 6: Validação forte contra o agente aleatório

Nesta secção verificamos automaticamente um dos critérios definidos no planeamento: cada modelo deve ser testado contra o agente `random` durante um número elevado de partidas e atingir uma taxa de vitória muito alta.

- `validation_games_against_random`: número de jogos usados nesta validação.
- `validation_threshold_against_random`: limiar mínimo de taxa de vitória para considerar o critério cumprido.
- A tabela final mostra, para cada modelo, o número de jogos, vitórias, derrotas, empates, taxa de vitória e se o critério foi cumprido.


In [ ]:
def make_random_reference_factory(base_seed: int):
    counter = {"value": 0}

    def factory():
        counter["value"] += 1
        return RandomAgent(seed=base_seed + counter["value"])

    return factory

validation_against_random = []

for model_name, factory in comparison_factories.items():
    if model_name == "random":
        continue

    random_factory = make_random_reference_factory(seed + 1000)
    duel_results = round_robin(
        {
            model_name: factory,
            "random": random_factory,
        },
        games_per_pair=validation_games_against_random,
    )
    metrics = duel_results[model_name]
    games_value = int(metrics.get("games", 0))
    wins_value = int(metrics.get("wins", 0))
    losses_value = int(metrics.get("losses", 0))
    draws_value = int(metrics.get("draws", 0))
    win_rate_value = float(metrics.get("win_rate", 0.0))
    criterion_met = games_value >= validation_games_against_random and win_rate_value >= validation_threshold_against_random
    validation_against_random.append({
        "modelo": model_name,
        "jogos": games_value,
        "vitórias": wins_value,
        "derrotas": losses_value,
        "empates": draws_value,
        "taxa_de_vitória": round(win_rate_value, 3),
        "critério_>=95%_em_200_jogos": "sim" if criterion_met else "não",
    })

validation_against_random.sort(key=lambda row: row["taxa_de_vitória"], reverse=True)

def _markdown_table(rows: list[dict]) -> str:
    headers = [
        "modelo",
        "jogos",
        "vitórias",
        "derrotas",
        "empates",
        "taxa_de_vitória",
        "critério_>=95%_em_200_jogos",
    ]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(str(row[h]) for h in headers) + " |")
    return "\n".join(lines)

print(_markdown_table(validation_against_random))
validation_against_random


## Passo 7: Cálculo de Elo do torneio final

Aqui estimamos o `Elo` dos agentes a partir do registo das partidas do torneio final.

In [ ]:
if comparison_matches:
    final_elo = compute_elo_ratings(comparison_matches)
    final_elo


## Passo 8: Interpretação automática para o relatório

Esta célula gera um resumo textual curto que ajuda a escrever a secção de conclusões experimentais do relatório.

In [ ]:
if comparison_results:
    strongest_name, strongest_stats = ranking[0]
    conclusion_lines = [
        f"Strongest model by tournament win rate: {strongest_name}",
        f"Win rate: {strongest_stats['win_rate']:.3f}",
        f"Draw rate: {strongest_stats['draw_rate']:.3f}",
    ]
    if comparison_matches:
        elo_top_name, elo_top_value = max(final_elo.items(), key=lambda item: item[1])
        conclusion_lines.append(f"Highest Elo: {elo_top_name} ({elo_top_value})")
    if "mcts" in comparison_results:
        conclusion_lines.append(f"MCTS win rate: {comparison_results['mcts']['win_rate']:.3f}")
    if "alphazero" in comparison_results:
        conclusion_lines.append(f"AlphaZero win rate: {comparison_results['alphazero']['win_rate']:.3f}")
    print("\n".join(conclusion_lines))


## Passo 9: Visualização de um confronto direto

Por fim, renderizamos uma partida concreta entre dois agentes relevantes para inspeção qualitativa.

In [ ]:
def play_and_render(agent, opponent, controlled_player: int = 1) -> str:
    state = initial_state()
    transcript = ["Initial board", render_ascii(state), ""]
    move_idx = 0
    while not is_terminal(state):
        move_idx += 1
        if state.current_player == controlled_player:
            action = agent.select_action(state, legal_actions(state))
            actor = agent.name
        else:
            action = opponent.select_action(state, legal_actions(state))
            actor = opponent.name
        state = apply_action(state, action)
        transcript.append(f"Move {move_idx}: {actor} played column {action}")
        transcript.append(render_ascii(state))
        transcript.append("")

    transcript.append(f"Winner: {state.winner}")
    return "\n".join(transcript)

if "mcts" in comparison_factories and "dqn" in comparison_factories:
    print(play_and_render(comparison_factories["mcts"](), comparison_factories["dqn"](), controlled_player=1))


## Fecho

Este notebook é o ponto final da análise experimental do projeto e deve ser usado para suportar a comparação entre métodos no relatório e na apresentação.
